# 第 1 章 · 项目总览与 uv 环境

> 📄 配套动画网页：`../01_overview.html`　（先用浏览器打开，把原理动画看一遍再来写代码）

**所属部分**：第一部分 · 基础

这一章不写模型，只做两件事：**看清 MiniMind 的全貌**，以及**把环境搭好**。后面 14 章都会在这套环境里跑。

---


## 🎯 本章目标

- 理解 MiniMind 的完整训练流水线（分词 → 预训练 → SFT → 对齐/RL）
- 掌握 `pathlib`、`dataclasses`、随机种子等工程基础
- 会用 uv 创建环境、安装依赖、启动 JupyterLab
- 能自己写出一个配置类并统计模型参数量

---


## 📖 背景：MiniMind 是什么

MiniMind 用极简代码实现了一个完整的语言模型训练链路，规模只有 **64M 参数**，
单张消费级显卡、几小时就能跑完。它的价值在于——**短到你能逐行读懂**。

| 阶段 | 脚本 | 做什么 |
|---|---|---|
| 分词 | train_tokenizer.py | 训练 6400 词表的 BPE |
| 预训练 | train_pretrain.py | 下一词预测，学语言规律 |
| 指令微调 | train_full_sft.py | 学会对话 / 问答 / 工具调用 |
| LoRA | train_lora.py | 冻结底座，只训低秩增量 |
| 蒸馏 | train_distillation.py | 用大模型当老师 |
| 偏好对齐 | train_dpo.py | chosen / rejected 对比 |
| 强化学习 | train_ppo / grpo.py | 用奖励优化策略 |
| 智能体 RL | train_agent.py | 多轮工具调用轨迹 |

> 记住主线：**预训练让模型会“接话”，SFT 让它会“听话”**，其余都是可选增强。

---



## 🖼 官方配图

![MiniMind 稠密模型结构（Transformer Decoder-Only，对齐 Qwen3）](../images/LLM-structure.jpg)

*MiniMind 稠密模型结构（Transformer Decoder-Only，对齐 Qwen3）*

![MiniMind MoE 模型结构（前馈层替换为专家混合）](../images/LLM-structure-moe.jpg)

*MiniMind MoE 模型结构（前馈层替换为专家混合）*

![各阶段数据集的构成与推荐组合](../images/dataset.jpg)

*各阶段数据集的构成与推荐组合*

![GPT-3 参数配置参考（MiniMind 的缩放对照）](../images/gpt3_config.png)

*GPT-3 参数配置参考（MiniMind 的缩放对照）*

![MiniMind 在第三方评测上的表现](../images/benchmark_radar.jpg)

*MiniMind 在第三方评测上的表现*

---


## 🖼 官方配图

![MiniMind 项目标识](../images/logo2.png)

*MiniMind 项目标识*

![模型权重也发布在 Hugging Face 上，可直接下载对照](../images/with_huggingface.png)

*模型权重也发布在 Hugging Face 上，可直接下载对照*

![国内可通过 ModelScope 获取权重与数据集](../images/with_modelscope.png)

*国内可通过 ModelScope 获取权重与数据集*

---


# 第一部分 · 基础函数演示

> 这一部分**可以直接运行**。先把本章要用到的 Python / PyTorch 函数和方法摸熟，后面自己写的时候才不会卡在 API 上。

下面这些是最基础的工程工具，写任何深度学习项目都会用到。


### 1.1 检查环境：`sys` 与版本信息

先确认 Python 和 PyTorch 的版本。深度学习对版本很敏感。

In [ ]:
import sys, platform
print("Python :", sys.version.split()[0])
print("平台   :", platform.platform())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA 可用:", torch.cuda.is_available())
    print("设备   :", "cuda" if torch.cuda.is_available() else "cpu")
except ImportError:
    print("PyTorch 未安装 —— 请在项目目录执行：uv sync")

### 1.2 路径处理：`pathlib.Path`

比字符串拼路径更安全，跨平台。深度学习项目里到处都要读写数据集、权重文件。

In [ ]:
from pathlib import Path

p = Path("..") / "notebooks"          # 用 / 拼路径
print("路径       :", p)
print("绝对路径   :", p.resolve())
print("是否存在   :", p.exists())
print("后缀       :", Path("model.pth").suffix)
print("文件名     :", Path("a/b/model.pth").stem)

### 1.3 配置对象：`dataclasses.dataclass`

MiniMind 的 `MiniMindConfig` 就是一个配置容器。`dataclass` 让你少写一堆 `__init__`。

In [ ]:
from dataclasses import dataclass, field
import math

@dataclass
class DemoConfig:
    hidden_size: int = 768
    num_hidden_layers: int = 8
    num_attention_heads: int = 8
    num_key_value_heads: int = 4
    vocab_size: int = 6400
    head_dim: int = field(init=False)

    def __post_init__(self):
        # 每个注意力头的维度 = hidden / heads
        self.head_dim = self.hidden_size // self.num_attention_heads

cfg = DemoConfig()
print(cfg)
print("head_dim =", cfg.head_dim)
# MiniMind 的 intermediate_size 会对齐到 64 的倍数，让矩阵乘法更快
print("intermediate_size =", math.ceil(768 * math.pi / 64) * 64)

### 1.4 随机种子：让实验可复现

同样的代码跑两次结果不同，是深度学习里最让人头疼的事之一。固定种子能解决大部分问题。

In [ ]:
import random
import numpy as np
import torch

def setup_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

setup_seed(42)
a = torch.randn(3)
setup_seed(42)
b = torch.randn(3)
print("两次随机结果是否一致:", torch.allclose(a, b))

# 第二部分 · 动手实现（只给提示，留白给你）

> 下面每一节都**没有答案**。请根据文字提示 + 第一部分演示过的函数，自己从零写出来。
> 写不出来时，先回去看第一部分的演示，再对照网页里的动画想清楚数据形状。

下面两件事是本章要你独立完成的。提示足够，答案不给。

> 💡 每个留白代码块后面都跟了一个**测试块**。先写代码，再运行测试块；
> 没写之前运行也不会报错，只会提示「未实现」。


In [ ]:
# ════════════════════ 自测工具 ════════════════════
# 每个留白代码块后面都跟一个测试块。先在上面写代码，再运行测试块。
# 没写之前运行也不会报错，只会提示「未实现」。
import torch
import torch.nn.functional as F

_PASS, _FAIL, _SKIP = [], [], []

def check(name, fn):
    """运行一个测试。捕获异常并打印结果，不会中断后面的测试。"""
    try:
        fn()
        print(f"✅ 通过   {name}")
        _PASS.append(name)
    except NameError as e:
        print(f"⏸  未实现 {name} —— {e}")
        _SKIP.append(name)
    except AssertionError as e:
        print(f"❌ 失败   {name} —— {e}")
        _FAIL.append(name)
    except Exception as e:
        print(f"❌ 出错   {name} —— {type(e).__name__}: {e}")
        _FAIL.append(name)

def close(a, b, tol=1e-4, msg=""):
    """断言两个张量 / 数字足够接近"""
    a, b = torch.as_tensor(a).float().reshape(-1), torch.as_tensor(b).float().reshape(-1)
    assert a.numel() == b.numel(), f"元素个数不一致 {a.numel()} vs {b.numel()} {msg}"
    assert torch.allclose(a, b, atol=tol), f"数值不接近（容差 {tol}）{msg}"

def same(a, b, msg=""):
    """断言两个张量完全相同"""
    a, b = torch.as_tensor(a), torch.as_tensor(b)
    assert a.shape == b.shape, f"形状不一致 {tuple(a.shape)} vs {tuple(b.shape)} {msg}"
    assert torch.equal(a, b), f"内容不相同 {msg}"

def summary():
    print()
    print(f"通过 {len(_PASS)} · 失败 {len(_FAIL)} · 未实现 {len(_SKIP)}")
    if _FAIL: print("需要修的：", "、".join(_FAIL))
    if _SKIP: print("还没写的：", "、".join(_SKIP))
    if _PASS and not _FAIL and not _SKIP:
        print("本章全部通过 🎉")

print("自测工具已就绪 —— 每写完一个留白，就运行它下面的测试块。")

### 2.1 写一个 `MiniMindConfig`

参考上面的 `DemoConfig`，写出一个更完整的配置类。

**要求**：包含 `hidden_size=768`、`num_hidden_layers=8`、`num_attention_heads=8`、
`num_key_value_heads=4`、`vocab_size=6400`、`rope_theta=1e6`、`rms_norm_eps=1e-6`、
`tie_word_embeddings=True`，并在 `__post_init__` 里算出 `head_dim` 和 `intermediate_size`。

In [ ]:
# ======== 你的实现：MiniMindConfig ========
#
# 【要求】
#   - 字段：hidden_size / num_hidden_layers / num_attention_heads / num_key_value_heads
#   - 字段：vocab_size / rope_theta / rms_norm_eps / tie_word_embeddings
#   - 在 __post_init__ 中计算 head_dim = hidden_size // num_attention_heads
#   - 在 __post_init__ 中计算 intermediate_size = ceil(hidden_size * π / 64) * 64
#
# 【可用到的函数 / 方法】
#   - `@dataclass`、`field(init=False)`
#   - `math.ceil`、`math.pi`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：MiniMindConfig ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    c = MiniMindConfig()
    assert c.hidden_size == 768, f"hidden_size 应为 768，得到 {c.hidden_size}"
    assert c.num_hidden_layers == 8, f"num_hidden_layers 应为 8，得到 {c.num_hidden_layers}"
    assert c.head_dim == 96, f"head_dim 应为 768 // 8 = 96，得到 {c.head_dim}"
    assert c.intermediate_size == 2432, f"intermediate_size 应为 ceil(768*pi/64)*64 = 2432，得到 {c.intermediate_size}"
    assert c.rope_theta == 1e6, f"rope_theta 应为 1e6，得到 {c.rope_theta}"
    assert c.tie_word_embeddings is True, "tie_word_embeddings 应为 True"

check("MiniMindConfig 字段与推导值", _t)

### 2.2 参数量统计函数

写一个函数，统计一个 `nn.Module` 的总参数量（单位：百万 M），并分别打印每个子模块的参数量。

**提示**：用 `model.named_children()` 或 `model.named_modules()` 遍历，
每个参数张量的元素个数是 `p.numel()`。

In [ ]:
# ======== 你的实现：count_parameters ========
#
# 【要求】
#   - 输入：一个 nn.Module
#   - 返回：总参数量（整数，即所有参数元素个数之和）
#   - 额外：打印每个直接子模块的参数量，以及总量（以 M 为单位，保留 2 位小数）
#
# 【可用到的函数 / 方法】
#   - `sum(p.numel() for p in model.parameters())`
#   - `model.named_children()`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：count_parameters ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    from torch import nn
    m = nn.Sequential(nn.Linear(4, 8, bias=False), nn.Linear(8, 2, bias=False))
    n = count_parameters(m)
    assert n == 4 * 8 + 8 * 2, f"参数量应为 48，得到 {n}"

check("count_parameters 统计正确", _t)

### 2.3 uv 常用命令（在终端执行，不是在这里）

```bash
uv sync                 # 按 pyproject.toml 创建/更新环境
uv run jupyter lab      # 在项目环境里启动 JupyterLab
uv add matplotlib       # 加一个依赖
uv add --dev pytest     # 加开发依赖
uv sync --extra train   # 安装可选依赖组（训练用）
uv lock --upgrade       # 升级依赖并重写锁文件
```

> 为什么用 uv：它比 pip 快一个数量级，并且 `uv.lock` 保证你和别人装出**完全一致**的版本。

In [ ]:
summary()

# 第三部分 · 自测与思考

- [ ] 能说清 MiniMind 的 8 个训练阶段以及它们的先后顺序
- [ ] 能解释为什么需要固定随机种子
- [ ] `MiniMindConfig` 的 `head_dim` 和 `intermediate_size` 算对了吗？
- [ ] 会用 `uv sync` / `uv run` 管理本项目环境

**思考题**：如果把 `num_attention_heads` 从 8 改成 16，`head_dim` 会变成多少？这会影响注意力的计算量吗？
